# Generating reference points for a group of decision makers

When each decision maker (DM) in a group gives an aspiration point, `desdeo.tools.group_reference_points` generates
reference points inside the convex hull of those points. Passing them to the Iterative Pareto Representer (IPR, see
[How to generate a representative set of solutions](../IPR/)) gives Pareto optimal solutions that lie between the DMs'
aspirations. This notebook shows one such round on a problem with three objectives.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import polars as pl
from IPython.display import clear_output

from desdeo.problem.testproblems import forest_problem
from desdeo.tools import GurobipySolver, payoff_table_method
from desdeo.tools.generateReferencePoints import generate_points
from desdeo.tools.group_reference_points import (
    denormalize_reference_point,
    generate_group_reference_points,
    normalize_objective_vectors,
    project_to_reference_plane,
)
from desdeo.tools.iterative_pareto_representer import _EvaluatedPoint, choose_reference_point
from desdeo.tools.scalarization import add_asf_diff

The problem is the forest management problem from the IPR guide. Its three objectives are all maximized: net present
value (`f_1`), wood stock volume (`f_2`) and harvest value (`f_3`). The ideal and nadir points are needed to normalize
the aspirations, and here they come from the payoff table.

In [ ]:
problem = forest_problem(
    simulation_results="../../tests/data/alternatives_290124.csv",
    treatment_key="../../tests/data/alternatives_key_290124.csv",
    holding=5,
    comparing=True,
)
ideal, nadir = payoff_table_method(problem=problem)
clear_output()
problem = problem.update_ideal_and_nadir(new_ideal=ideal, new_nadir=nadir)
symbols = [objective.symbol for objective in problem.objectives]

pl.DataFrame([ideal, nadir]).insert_column(0, pl.Series("point", ["ideal", "nadir"]))

Each DM gives an aspiration point in the original units. Several DMs may give the same point, and some points may
lie inside the hull of the others. Here DM 5 repeats DM 1.

In [ ]:
aspirations = [
    {"f_1": 70000.0, "f_2": -1500.0, "f_3": 130000.0},
    {"f_1": 35000.0, "f_2": 800.0, "f_3": 30000.0},
    {"f_1": 60000.0, "f_2": 500.0, "f_3": 60000.0},
    {"f_1": 55000.0, "f_2": 0.0, "f_3": 75000.0},
    {"f_1": 70000.0, "f_2": -1500.0, "f_3": 130000.0},
]
dm_names = [f"DM {i + 1}" for i in range(len(aspirations))]

pl.DataFrame(aspirations).insert_column(0, pl.Series("DM", dm_names))

`generate_group_reference_points` returns the candidate reference points for IPR. They are in IPR's normalized
space (ideal at 0, nadir at 1, every objective minimized) and each row sums to the number of objectives.
`denormalize_reference_point` converts a row back to original units. The candidates lie on a plane through the
nadir, which is where IPR expects them, so in original units a candidate can be worse than the nadir in some
objectives, as in the example below.

In [ ]:
reference_points = generate_group_reference_points(problem, aspirations, num_points=10_000, seed=0)
print(f"Shape: {reference_points.shape}")
denormalize_reference_point(problem, reference_points[0])

The IPR loop is the one from the IPR guide, wrapped in a function so that it can also run over plain IPR's
candidates below. IPR discards candidates that would lead back to a solution it has already found, and
`choose_reference_point` raises an error once none are left. A hull this size supports the 30 iterations used
for the group.

In [ ]:
def run_ipr(candidates: np.ndarray, num_iterations: int) -> list[_EvaluatedPoint]:
    """Run IPR over normalized candidate reference points and return the evaluated points."""
    # choose_reference_point picks the first candidate with NumPy's global generator.
    np.random.seed(0)  # noqa: NPY002
    evaluated_points: list[_EvaluatedPoint] = []
    for _ in range(num_iterations):
        reference_point, _ = choose_reference_point(candidates, evaluated_points or None)
        scalarized, target = add_asf_diff(problem, "asf", denormalize_reference_point(problem, reference_point))
        objectives = GurobipySolver(scalarized, options={"OutputFlag": 0}).solve(target).optimal_objectives
        targets = normalize_objective_vectors(problem, [objectives])[0]
        evaluated_points.append(
            _EvaluatedPoint(
                reference_point=dict(zip(symbols, reference_point.tolist(), strict=True)),
                targets=dict(zip(symbols, targets.tolist(), strict=True)),
                objectives=objectives,
            )
        )
    return evaluated_points


group_points = run_ipr(reference_points, num_iterations=30)

For reference, the same loop over the candidates of plain IPR, which `generate_points` spreads over the whole
plane, gives a representation of the entire Pareto front.

In [ ]:
_, plain_candidates = generate_points(num_points=10_000, num_dims=len(symbols))
front_points = run_ipr(plain_candidates, num_iterations=100)

The figure is in original units. It shows the DMs' aspirations and their projections onto the IPR plane, joined
by dashed lines, the reference points generated for the group, the ones IPR evaluated, the solutions found from
them, and the Pareto front for reference. The projections and the reference points lie on the plane through the
nadir, away from the front, so rotate and zoom to see both. Hovering shows the values.

In [ ]:
def to_original_units(points: np.ndarray) -> np.ndarray:
    """Convert normalized points, one per row, to original units."""
    return np.array([list(denormalize_reference_point(problem, point).values()) for point in points])


def objective_vectors(evaluated_points: list[_EvaluatedPoint]) -> np.ndarray:
    """The objective vectors of evaluated points, one per row."""
    return np.array([list(point.objectives.values()) for point in evaluated_points])


def scatter3d(points: np.ndarray, **kwargs) -> go.Scatter3d:
    """A 3D scatter trace of points given one per row."""
    return go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], **kwargs)


# Identical aspirations share one marker and one label.
groups: dict[tuple, list[str]] = {}
for name, aspiration in zip(dm_names, aspirations, strict=True):
    groups.setdefault(tuple(aspiration.values()), []).append(name)
aspiration_points = np.array(list(groups))
aspiration_labels = [", ".join(names) for names in groups.values()]
projections = to_original_units(
    project_to_reference_plane(
        normalize_objective_vectors(problem, [dict(zip(symbols, point, strict=True)) for point in aspiration_points])
    )
)
projection_lines = np.full((3 * len(aspiration_points), 3), np.nan)
projection_lines[0::3] = aspiration_points
projection_lines[1::3] = projections
evaluated_reference_points = np.array([list(point.reference_point.values()) for point in group_points])

fig = go.Figure()
fig.add_trace(
    scatter3d(
        objective_vectors(front_points),
        mode="markers",
        marker={"size": 3, "color": "#898781"},
        name="Pareto front (plain IPR)",
    )
)
fig.add_trace(
    scatter3d(
        to_original_units(reference_points),
        mode="markers",
        marker={"size": 1.5, "color": "#c3c2b7"},
        name="Reference points generated for the group",
        hoverinfo="skip",
    )
)
fig.add_trace(
    scatter3d(
        to_original_units(evaluated_reference_points),
        mode="markers",
        marker={"size": 4, "color": "#eb6834", "symbol": "diamond"},
        name="Reference points evaluated by IPR",
    )
)
fig.add_trace(
    scatter3d(
        objective_vectors(group_points),
        mode="markers",
        marker={"size": 5, "color": "#eb6834"},
        name="Solutions for the group",
    )
)
fig.add_trace(
    scatter3d(
        projection_lines,
        mode="lines",
        line={"color": "#2a78d6", "width": 2, "dash": "dash"},
        name="Projection onto the IPR plane",
        hoverinfo="skip",
    )
)
fig.add_trace(
    scatter3d(
        projections,
        mode="markers",
        hovertext=aspiration_labels,
        marker={"size": 5, "color": "#2a78d6", "symbol": "circle-open"},
        name="Projected aspirations",
    )
)
fig.add_trace(
    scatter3d(
        aspiration_points,
        mode="markers+text",
        text=aspiration_labels,
        marker={"size": 6, "color": "#2a78d6"},
        name="DM aspirations",
    )
)
fig.update_layout(
    template="plotly_white",
    height=700,
    margin={"l": 0, "r": 0, "t": 10, "b": 0},
    legend={"orientation": "h"},
    scene={f"{axis}axis_title": objective.name for axis, objective in zip("xyz", problem.objectives, strict=True)},
)
fig.layout.scene.camera.projection.type = "orthographic"
fig.show(renderer="notebook")

The solutions for the group as a table:

In [ ]:
pl.DataFrame([point.objectives for point in group_points])